# Step 6 — Why Better Tracks Matter: Route Analytics

Once reconstruction works, we use the resulting trajectories for downstream analytics:

1. **Route distance** — how far did the aircraft actually fly?
2. **Deviation from expected path** — how far off the great-circle was the real route?
3. **CO₂ emissions proxy** — how much fuel was burned?

The key insight: if you use only raw ADS-B (which has a gap over the ocean), you **underestimate** the true route distance. Better reconstruction gives more accurate downstream analytics.

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6

## Cell 1 — Setup

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pyproj import Geod
from scipy.interpolate import interp1d
from tqdm import tqdm

# ── Point to noel_part ────────────────────────────────────────────────────────
NOEL_DIR = Path(r"C:\Users\marko\Desktop\AeroML3\noel_part")
if not NOEL_DIR.exists():
    for _c in [Path("../noel_part"), Path("../../noel_part")]:
        if _c.resolve().exists(): NOEL_DIR = _c.resolve(); break

os.chdir(NOEL_DIR)
sys.path.insert(0, str(NOEL_DIR))

BASE_DIR  = Path(".")
CLEAN_DIR = BASE_DIR / "cleaned_data_final"
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

geod = Geod(ellps="WGS84")

from reconstruction import (TrajectoryBiGRU, FEATURE_COLS, TARGET_COLS,
                             SEQUENCE_LEN, reconstruct_gap_kalman,
                             reconstruct_gap, compute_features)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TrajectoryBiGRU(len(FEATURE_COLS), 128, 2, len(TARGET_COLS), 0.3).to(device)
model.load_state_dict(torch.load("models/BiGRU.pth", map_location=device))
model.eval()

flights = [f for s in sorted(CLEAN_DIR.iterdir()) if s.is_dir()
           for f in sorted(s.iterdir()) if f.is_dir()]

# Widebody aircraft fuel burn proxy (kg CO2 per km flown)
# Based on ICAO average: ~6 kg fuel/km × 3.16 CO2 factor ≈ 19 kg CO2/km
CO2_KG_PER_KM = 19.0

print(f"Working dir   : {os.getcwd()}")
print(f"Device        : {device}")
print(f"Total flights : {len(flights)}")
print(f"CO2 proxy     : {CO2_KG_PER_KM} kg CO2/km")

## Cell 2 — Helper functions

In [ ]:
def reconstruct_forward_only(before_df, after_df, dt=60.0):
    last_time  = before_df["timestamp"].iloc[-1]
    first_time = after_df["timestamp"].iloc[0]
    n_steps = max(1, int(round((first_time - last_time).total_seconds() / dt)))
    lat0 = float(before_df["latitude"].iloc[-1]); lon0 = float(before_df["longitude"].iloc[-1])
    alt0 = float(before_df["altitude"].iloc[-1])
    lat1 = float(after_df["latitude"].iloc[0]);   lon1 = float(after_df["longitude"].iloc[0])
    alt1 = float(after_df["altitude"].iloc[0])
    pts  = geod.npts(lon0, lat0, lon1, lat1, n_steps)
    lats = np.array([p[1] for p in pts]); lons = np.array([p[0] for p in pts])
    alts = np.linspace(alt0, alt1, n_steps)
    timestamps = [last_time + pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    r = pd.DataFrame({"latitude": lats, "longitude": lons, "altitude": alts})
    r["timestamp"] = timestamps; r["interpolated"] = True
    return r

def retime_to_constant_speed(recon_df, before_df, after_df, dt=60.0):
    lats = recon_df["latitude"].values; lons = recon_df["longitude"].values
    alts = recon_df["altitude"].values if "altitude" in recon_df.columns else np.zeros(len(recon_df))
    cum_dist = np.zeros(len(lats))
    for i in range(1, len(lats)):
        _, _, d = geod.inv(float(lons[i-1]), float(lats[i-1]), float(lons[i]), float(lats[i]))
        cum_dist[i] = cum_dist[i-1] + abs(d)
    total_dist = cum_dist[-1]
    if total_dist == 0: return recon_df
    last_time  = before_df["timestamp"].iloc[-1]
    first_time = after_df["timestamp"].iloc[0]
    total_sec  = (first_time - last_time).total_seconds()
    n_steps    = max(1, int(round(total_sec / dt)))
    time_fracs = np.clip([dt*(i+1)/total_sec for i in range(n_steps)], 0, 1)
    old_fracs  = cum_dist / total_dist
    f_la  = interp1d(old_fracs, lats, bounds_error=False, fill_value=(lats[0], lats[-1]))
    f_lo  = interp1d(old_fracs, lons, bounds_error=False, fill_value=(lons[0], lons[-1]))
    f_alt = interp1d(old_fracs, alts, bounds_error=False, fill_value=(alts[0], alts[-1]))
    new_ts = [last_time + pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    return pd.DataFrame({"latitude": f_la(time_fracs), "longitude": f_lo(time_fracs),
                         "altitude": f_alt(time_fracs), "timestamp": new_ts, "interpolated": True})

def path_length_km(df, lat_col="latitude", lon_col="longitude"):
    """Total geodesic path length in km."""
    lats = df[lat_col].values; lons = df[lon_col].values
    finite = np.isfinite(lats) & np.isfinite(lons)
    lats = lats[finite]; lons = lons[finite]
    if len(lats) < 2: return 0.0
    _, _, dists = geod.inv(lons[:-1], lats[:-1], lons[1:], lats[1:])
    return float(np.nansum(np.abs(dists))) / 1000

def max_deviation_km(recon_df, start_lat, start_lon, end_lat, end_lon):
    """
    Maximum lateral deviation from the great-circle reference path (km).
    This measures how far the reconstruction deviates from the expected route.
    """
    if len(recon_df) == 0: return 0.0
    deviations = []
    brg_ref = np.radians(
        (np.degrees(np.arctan2(
            np.sin(np.radians(end_lon-start_lon))*np.cos(np.radians(end_lat)),
            np.cos(np.radians(start_lat))*np.sin(np.radians(end_lat)) -
            np.sin(np.radians(start_lat))*np.cos(np.radians(end_lat))*
            np.cos(np.radians(end_lon-start_lon))
        )) + 360) % 360
    )
    R = 6_371_000.0
    for _, row in recon_df.iterrows():
        la1,lo1 = np.radians(start_lat), np.radians(start_lon)
        la2,lo2 = np.radians(row["latitude"]), np.radians(row["longitude"])
        dlat,dlon = la2-la1, lo2-lo1
        a = np.sin(dlat/2)**2 + np.cos(la1)*np.cos(la2)*np.sin(dlon/2)**2
        d13 = 2*np.arcsin(np.sqrt(np.clip(a,0,1)))
        brg13 = np.arctan2(np.sin(dlon)*np.cos(la2),
                            np.cos(la1)*np.sin(la2)-np.sin(la1)*np.cos(la2)*np.cos(dlon))
        xte = np.arcsin(np.clip(np.sin(d13)*np.sin(brg13-brg_ref),-1,1))*R
        deviations.append(abs(xte))
    return float(np.max(deviations)) / 1000 if deviations else 0.0

print("Helper functions defined.")

## Cell 3 — Compute analytics for all flights

In [ ]:
records = []
skipped = 0
DT = 60.0  # 60s step — fast enough for route analytics

print(f"Computing route analytics for {len(flights)} flights ...")

for flight_dir in tqdm(flights, desc="Analytics"):
    try:
        _bp  = next((flight_dir/p for p in ["adsb_before_clean.parquet","adsb_before.parquet"]
                     if (flight_dir/p).exists()), None)
        _afp = next((flight_dir/p for p in ["adsb_after_clean.parquet","adsb_after.parquet"]
                     if (flight_dir/p).exists()), None)
        _ap  = next((flight_dir/p for p in ["adsc_clean.parquet","adsc.parquet"]
                     if (flight_dir/p).exists()), None)

        if not (_bp and _afp): skipped += 1; continue

        bef = pd.read_parquet(str(_bp)).dropna(subset=["latitude","longitude"])
        aft = pd.read_parquet(str(_afp)).dropna(subset=["latitude","longitude"])

        for df in [bef, aft]:
            df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_localize(None)
            df.sort_values("timestamp", inplace=True)
            df.reset_index(drop=True, inplace=True)

        if len(bef) < 4 or len(aft) < 4: skipped += 1; continue

        t_gap_start = bef["timestamp"].iloc[-1]
        t_gap_end   = aft["timestamp"].iloc[0]
        gap_min = (t_gap_end - t_gap_start).total_seconds() / 60
        if gap_min < 5: skipped += 1; continue

        # Trim to boundary context
        bef_trim = bef.tail(max(SEQUENCE_LEN, 15)).reset_index(drop=True)
        aft_trim = aft.head(max(SEQUENCE_LEN, 15)).reset_index(drop=True)
        if len(bef_trim) < 2 or len(aft_trim) < 2: skipped += 1; continue

        start_lat = float(bef_trim["latitude"].iloc[-1])
        start_lon = float(bef_trim["longitude"].iloc[-1])
        end_lat   = float(aft_trim["latitude"].iloc[0])
        end_lon   = float(aft_trim["longitude"].iloc[0])

        # ── 1. Raw ADS-B distance (with gap — underestimates true distance) ──
        raw_full = pd.concat([bef, aft], ignore_index=True).sort_values("timestamp")
        raw_dist_km = path_length_km(raw_full)

        # ── 2. Great-circle straight-line gap distance ─────────────────────
        _, _, gc_gap_m = geod.inv(start_lon, start_lat, end_lon, end_lat)
        gc_gap_km = abs(gc_gap_m) / 1000

        # ── 3. Reconstruct the gap with all 3 methods ──────────────────────
        recon_base = reconstruct_forward_only(bef_trim, aft_trim, dt=DT)

        recon_kalman_raw = compute_features(reconstruct_gap_kalman(bef_trim, aft_trim, dt=DT))
        recon_kalman     = retime_to_constant_speed(recon_kalman_raw, bef_trim, aft_trim, dt=DT)

        recon_bigru_raw  = compute_features(
            reconstruct_gap(model, bef_trim, aft_trim, FEATURE_COLS, TARGET_COLS, SEQUENCE_LEN, device, dt=DT)
        )
        recon_bigru = retime_to_constant_speed(recon_bigru_raw, bef_trim, aft_trim, dt=DT)

        # ── 4. Full reconstructed route distance (ADS-B + gap) ─────────────
        def full_route_km(gap_recon):
            full = pd.concat([
                bef[["latitude","longitude"]],
                gap_recon[["latitude","longitude"]],
                aft[["latitude","longitude"]]
            ], ignore_index=True)
            return path_length_km(full)

        base_dist_km   = full_route_km(recon_base)
        kalman_dist_km = full_route_km(recon_kalman)
        bigru_dist_km  = full_route_km(recon_bigru)

        # ── 5. ADS-C true distance (if available) ──────────────────────────
        adsc_dist_km = np.nan
        if _ap:
            adsc = pd.read_parquet(str(_ap)).dropna(subset=["latitude","longitude"])
            adsc["timestamp"] = pd.to_datetime(adsc["timestamp"], utc=True).dt.tz_localize(None)
            adsc_gap = adsc[(adsc["timestamp"] > t_gap_start) &
                            (adsc["timestamp"] < t_gap_end)].reset_index(drop=True)
            if len(adsc_gap) >= 2:
                adsc_full = pd.concat([
                    bef[["latitude","longitude"]],
                    adsc_gap[["latitude","longitude"]],
                    aft[["latitude","longitude"]]
                ], ignore_index=True)
                adsc_dist_km = path_length_km(adsc_full)

        # ── 6. Max deviation from great-circle path ─────────────────────────
        base_dev_km   = max_deviation_km(recon_base,   start_lat, start_lon, end_lat, end_lon)
        kalman_dev_km = max_deviation_km(recon_kalman, start_lat, start_lon, end_lat, end_lon)
        bigru_dev_km  = max_deviation_km(recon_bigru,  start_lat, start_lon, end_lat, end_lon)

        records.append({
            "flight":            flight_dir.name,
            "step":              flight_dir.parent.name,
            "gap_minutes":       round(gap_min, 1),
            "gc_gap_km":         round(gc_gap_km, 1),

            # Route distances (km)
            "raw_adsb_dist_km":  round(raw_dist_km,   1),
            "baseline_dist_km":  round(base_dist_km,   1),
            "kalman_dist_km":    round(kalman_dist_km, 1),
            "bigru_dist_km":     round(bigru_dist_km,  1),
            "adsc_true_dist_km": round(adsc_dist_km, 1) if not np.isnan(adsc_dist_km) else np.nan,

            # Gap filled by each method (km)
            "baseline_gap_km":   round(path_length_km(recon_base),   1),
            "kalman_gap_km":     round(path_length_km(recon_kalman), 1),
            "bigru_gap_km":      round(path_length_km(recon_bigru),  1),

            # Max deviation from great-circle path (km)
            "baseline_dev_km":   round(base_dev_km,   1),
            "kalman_dev_km":     round(kalman_dev_km, 1),
            "bigru_dev_km":      round(bigru_dev_km,  1),

            # CO2 proxy (kg) — 19 kg CO2/km for widebody
            "raw_co2_kg":        round(raw_dist_km   * CO2_KG_PER_KM, 0),
            "baseline_co2_kg":   round(base_dist_km  * CO2_KG_PER_KM, 0),
            "kalman_co2_kg":     round(kalman_dist_km* CO2_KG_PER_KM, 0),
            "bigru_co2_kg":      round(bigru_dist_km * CO2_KG_PER_KM, 0),
            "adsc_co2_kg":       round(adsc_dist_km  * CO2_KG_PER_KM, 0) if not np.isnan(adsc_dist_km) else np.nan,
        })

    except Exception: skipped += 1; continue

analytics_df = pd.DataFrame(records)
analytics_df.to_csv(OUT_DIR / "analytics_results.csv", index=False)
print(f"\nFlights analysed : {len(analytics_df)}")
print(f"Flights skipped  : {skipped}")
print(f"Saved → {OUT_DIR / 'analytics_results.csv'}")

## Cell 4 — Route distance summary

Shows how much distance is **missed** by raw ADS-B vs each reconstruction method.

In [ ]:
print("=" * 65)
print("ROUTE ANALYTICS — Why Better Tracks Matter")
print("=" * 65)

print(f"\nFlights analysed : {len(analytics_df)}")
print(f"Avg gap duration : {analytics_df['gap_minutes'].mean():.1f} min")
print(f"Avg gap distance : {analytics_df['gc_gap_km'].mean():.0f} km (great-circle)")

print("\n── Route Distance (full flight: ADS-B before + gap + ADS-B after) ──")
print(f"{'Method':<20}  {'Mean dist (km)':>15}  {'vs Raw ADS-B':>13}  {'vs ADS-C truth':>15}")
print("-" * 68)

raw_mean   = analytics_df["raw_adsb_dist_km"].mean()
adsc_mean  = analytics_df["adsc_true_dist_km"].dropna().mean()

for method, col in [
    ("Raw ADS-B (gap)",  "raw_adsb_dist_km"),
    ("Baseline",         "baseline_dist_km"),
    ("Kalman",           "kalman_dist_km"),
    ("BiGRU",            "bigru_dist_km"),
    ("ADS-C truth",      "adsc_true_dist_km"),
]:
    vals = analytics_df[col].dropna()
    if len(vals) == 0: continue
    mean_val = vals.mean()
    vs_raw   = f"+{mean_val-raw_mean:+.0f} km" if method != "Raw ADS-B (gap)" else "—"
    vs_adsc  = f"{mean_val-adsc_mean:+.0f} km" if not np.isnan(adsc_mean) and method not in ["Raw ADS-B (gap)","ADS-C truth"] else "—"
    print(f"  {method:<18}  {mean_val:>15.0f}  {vs_raw:>13}  {vs_adsc:>15}")

print("\n── CO₂ Emissions Proxy (19 kg CO₂/km, widebody average) ──")
print(f"{'Method':<20}  {'Mean CO₂ (kg)':>14}  {'Extra CO₂ vs Raw':>18}")
print("-" * 57)
for method, col in [
    ("Raw ADS-B (gap)",  "raw_co2_kg"),
    ("Baseline",         "baseline_co2_kg"),
    ("Kalman",           "kalman_co2_kg"),
    ("BiGRU",            "bigru_co2_kg"),
    ("ADS-C truth",      "adsc_co2_kg"),
]:
    vals = analytics_df[col].dropna()
    if len(vals) == 0: continue
    mean_val = vals.mean()
    raw_co2  = analytics_df["raw_co2_kg"].mean()
    extra    = f"+{mean_val-raw_co2:+.0f} kg" if method != "Raw ADS-B (gap)" else "—"
    print(f"  {method:<18}  {mean_val:>14.0f}  {extra:>18}")

print("\n── Max Deviation from Great-Circle Path ──")
print(f"{'Method':<12}  {'Mean max deviation (km)':>24}")
print("-" * 40)
for method, col in [("Baseline","baseline_dev_km"),("Kalman","kalman_dev_km"),("BiGRU","bigru_dev_km")]:
    vals = analytics_df[col].dropna()
    print(f"  {method:<10}  {vals.mean():>24.1f}")

## Cell 5 — Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: Route distance comparison ─────────────────────────────────────────
ax = axes[0]
methods_dist = [
    ("Raw ADS-B\n(gap)", "raw_adsb_dist_km",  "#AAAAAA"),
    ("Baseline",          "baseline_dist_km",   "#F44336"),
    ("Kalman",            "kalman_dist_km",      "#00BCD4"),
    ("BiGRU",             "bigru_dist_km",       "#4CAF50"),
]
if analytics_df["adsc_true_dist_km"].notna().sum() > 0:
    methods_dist.append(("ADS-C\nTruth", "adsc_true_dist_km", "#FFC107"))

labels = [m[0] for m in methods_dist]
means  = [analytics_df[m[1]].dropna().mean() for m in methods_dist]
colors = [m[2] for m in methods_dist]

bars = ax.bar(labels, means, color=colors, edgecolor="white", width=0.6)
for bar, val in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            f"{val:.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_ylabel("Mean route distance (km)")
ax.set_title("Route Distance by Method")
ax.set_ylim(0, max(means)*1.15)
ax.grid(axis="y", alpha=0.3)

# ── Plot 2: CO2 proxy ─────────────────────────────────────────────────────────
ax = axes[1]
methods_co2 = [
    ("Raw ADS-B\n(gap)", "raw_co2_kg",      "#AAAAAA"),
    ("Baseline",          "baseline_co2_kg",  "#F44336"),
    ("Kalman",            "kalman_co2_kg",     "#00BCD4"),
    ("BiGRU",             "bigru_co2_kg",      "#4CAF50"),
]
if analytics_df["adsc_co2_kg"].notna().sum() > 0:
    methods_co2.append(("ADS-C\nTruth", "adsc_co2_kg", "#FFC107"))

labels2 = [m[0] for m in methods_co2]
means2  = [analytics_df[m[1]].dropna().mean() for m in methods_co2]
colors2 = [m[2] for m in methods_co2]

bars2 = ax.bar(labels2, means2, color=colors2, edgecolor="white", width=0.6)
for bar, val in zip(bars2, means2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
            f"{val:.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_ylabel("Mean CO₂ estimate (kg)")
ax.set_title("CO₂ Emissions Proxy by Method")
ax.set_ylim(0, max(means2)*1.15)
ax.grid(axis="y", alpha=0.3)

# ── Plot 3: Gap filled vs gap duration ────────────────────────────────────────
ax = axes[2]
ax.scatter(analytics_df["gap_minutes"], analytics_df["baseline_gap_km"],
           alpha=0.3, s=15, color="#F44336", label="Baseline")
ax.scatter(analytics_df["gap_minutes"], analytics_df["kalman_gap_km"],
           alpha=0.3, s=15, color="#00BCD4", label="Kalman")
ax.scatter(analytics_df["gap_minutes"], analytics_df["bigru_gap_km"],
           alpha=0.3, s=15, color="#4CAF50", label="BiGRU")
ax.scatter(analytics_df["gap_minutes"], analytics_df["gc_gap_km"],
           alpha=0.5, s=15, color="#AAAAAA", label="Great-circle (reference)")

ax.set_xlabel("Gap duration (minutes)")
ax.set_ylabel("Gap filled (km)")
ax.set_title("Distance Filled vs Gap Duration")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle("Why Better Reconstruction Matters: Route & Emissions Impact",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "analytics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {OUT_DIR / 'analytics_comparison.png'}")

## Cell 6 — Key insight: underestimation problem

In [ ]:
# ── How much does raw ADS-B underestimate the true route? ─────────────────────
gap_col = analytics_df["gc_gap_km"]
raw_col = analytics_df["raw_adsb_dist_km"]
base_col = analytics_df["baseline_dist_km"]

underest_km  = (base_col - raw_col).mean()
underest_pct = (underest_km / base_col.mean()) * 100

print("KEY FINDING: The Raw ADS-B Underestimation Problem")
print("=" * 55)
print(f"\nRaw ADS-B misses the oceanic gap entirely.")
print(f"Average gap distance      : {gap_col.mean():.0f} km")
print(f"Average route (raw ADS-B) : {raw_col.mean():.0f} km")
print(f"Average route (baseline)  : {base_col.mean():.0f} km")
print(f"\nUnderestimation           : {underest_km:.0f} km ({underest_pct:.1f}% of route)")
print(f"CO₂ underestimate         : {underest_km*CO2_KG_PER_KM:.0f} kg per flight")
print(f"CO₂ underestimate (fleet) : {underest_km*CO2_KG_PER_KM*365:.0f} kg/year per daily flight")

print("\n── Why this matters ──")
print(f"  A fleet of 100 long-haul aircraft each flying daily:")
print(f"  Missing CO₂  = {underest_km*CO2_KG_PER_KM*365*100/1e6:.1f} million kg CO₂/year unaccounted")
print(f"  Better reconstruction fixes this tracking gap.")

print("\n── Max route deviation from great-circle ──")
print(f"  Baseline: {analytics_df['baseline_dev_km'].mean():.1f} km  (straight arc, no deviation by design)")
print(f"  Kalman  : {analytics_df['kalman_dev_km'].mean():.1f} km  (slight deviation from velocity drift)")
print(f"  BiGRU   : {analytics_df['bigru_dev_km'].mean():.1f} km  (learned deviations from training data)")
print(f"\n  Real flights deviate from great-circle to follow jet streams.")
print(f"  A model that captures this deviation gives more realistic route analytics.")